In [1]:
import arxiv
import os

# Create directory for PDFs
os.makedirs("arxiv_papers", exist_ok=True)

# Search for 50 cs.CL papers and download them
search = arxiv.Search(
    query="cat:cs.CL",
    max_results=50,
    sort_by=arxiv.SortCriterion.SubmittedDate
)

pdf_files = []
print("Downloading 50 arXiv cs.CL papers...")
for i, result in enumerate(search.results()):
    pdf_path = f"arxiv_papers/{result.get_short_id()}.pdf"
    if not os.path.exists(pdf_path):
        result.download_pdf(filename=pdf_path)
    pdf_files.append(pdf_path)
    print(f"Downloaded {i+1}/50: {result.title[:60]}...")

print(f"\nTotal PDFs downloaded: {len(pdf_files)}")

C:\Users\Kevin\AppData\Local\Temp\ipykernel_21820\3019294951.py:16: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for i, result in enumerate(search.results()):


Downloaded 1/50: Generalist Foundation Models Are Not Clinical Enough for Hos...
Downloaded 2/50: Crossing Borders: A Multimodal Challenge for Indian Poetry T...
Downloaded 3/50: Why is "Chicago" Predictive of Deceptive Reviews? Using LLMs...
Downloaded 2/50: Crossing Borders: A Multimodal Challenge for Indian Poetry T...
Downloaded 3/50: Why is "Chicago" Predictive of Deceptive Reviews? Using LLMs...
Downloaded 4/50: Live-SWE-agent: Can Software Engineering Agents Self-Evolve ...
Downloaded 4/50: Live-SWE-agent: Can Software Engineering Agents Self-Evolve ...
Downloaded 5/50: P1: Mastering Physics Olympiads with Reinforcement Learning...
Downloaded 5/50: P1: Mastering Physics Olympiads with Reinforcement Learning...
Downloaded 6/50: Omni Memory System for Personalized, Long Horizon, Self-Evol...
Downloaded 6/50: Omni Memory System for Personalized, Long Horizon, Self-Evol...
Downloaded 7/50: Beyond SELECT: A Comprehensive Taxonomy-Guided Benchmark for...
Downloaded 7/50: Beyond SELECT

In [2]:
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Open a PDF and extract all text as a single string.
    """
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        page_text = page.get_text()  # get raw text from page
        
        # (Optional) clean page_text here (remove headers/footers)
        pages.append(page_text)
    full_text = "\n".join(pages)
    return full_text

In [3]:
from typing import List
def chunk_text(text: str, max_tokens: int = 512, overlap: int = 50) -> List[str]:
    tokens = text.split()
    chunks = []
    step = max_tokens - overlap
    for i in range(0, len(tokens), step):
        chunk = tokens[i:i + max_tokens]
        chunks.append(" ".join(chunk))
    return chunks


In [4]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Process all PDFs: extract, chunk, collect
all_chunks = []

for pdf_file in pdf_files:
    full_text = extract_text_from_pdf(pdf_file)
    chunks = chunk_text(full_text)
    all_chunks.extend(chunks)

# Generate embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(all_chunks)

# Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.array(embeddings))

c:\Users\Kevin\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Kevin\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kevin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as

In [5]:
# Save FAISS index and processed data
import pickle
import json

# Save FAISS index
faiss.write_index(index, "faiss_index.bin")

# Save chunks as JSON
with open("processed_chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

# Save all data as pickle for FastAPI
with open("rag_data.pkl", "wb") as f:
    pickle.dump({"all_chunks": all_chunks, "index": index}, f)

print(f"✅ Saved FAISS index to 'faiss_index.bin'")
print(f"✅ Saved {len(all_chunks)} chunks to 'processed_chunks.json'")
print(f"✅ Saved pickled data to 'rag_data.pkl'")

✅ Saved FAISS index to 'faiss_index.bin'
✅ Saved 981 chunks to 'processed_chunks.json'
✅ Saved pickled data to 'rag_data.pkl'


In [6]:
# Retrieval Report: 5 example queries with top-3 results each
queries = [
    "What are transformer models in natural language processing?",
    "How does attention mechanism work?",
    "What is BERT and how is it trained?",
    "Explain neural machine translation",
    "What are the applications of large language models?"
]

report = []
print("RETRIEVAL REPORT")
print("="*80)

for query in queries:
    print(f"\nQuery: {query}")
    print("-"*80)
    
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k=3)
    
    query_results = {"query": query, "results": []}
    
    for i, idx in enumerate(indices[0]):
        result_text = all_chunks[idx]
        distance = float(distances[0][i])
        
        print(f"\n[Result {i+1}] Distance: {distance:.4f}")
        print(f"Text: {result_text[:300]}...")
        
        query_results["results"].append({
            "rank": i+1,
            "distance": distance,
            "text": result_text
        })
    
    report.append(query_results)
    print()

# Save report
with open("retrieval_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("="*80)
print("✅ Report saved to 'retrieval_report.json'")

RETRIEVAL REPORT

Query: What are transformer models in natural language processing?
--------------------------------------------------------------------------------

[Result 1] Distance: 0.9190
Text: pre-trained on a large corpus of text. GPT-2 Generative Pre-trained Transformer 2. An autoregressive language model that uses unidirec- tional attention (each token can only attend to previous tokens). It contains 124 million parameters in its base version and was pre-trained on a larger corpus than...

[Result 2] Distance: 0.9381
Text: and Zhang, Y. 2022. Measuring and Reducing Model Update Regression in Structured Prediction for NLP. arXiv:2202.02976. Chang, J. P.; Chiam, C.; Fu, L.; Wang, A. Z.; Zhang, J.; and Danescu-Niculescu-Mizil, C. 2020. Convokit: A toolkit for the analysis of conversations. arXiv preprint arXiv:2005.04246...

[Result 3] Distance: 0.9784
Text: Classification of Hope in Textual Data using Transformer-Based Models Chukwuebuka Fortunate Ijezue1, Fredrick Eneye Tania

In [ ]:
# Save FastAPI app as main.py
app_code = '''from fastapi import FastAPI
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer

# Load saved data
with open("rag_data.pkl", "rb") as f:
    data = pickle.load(f)
    all_chunks = data["all_chunks"]
    index = data["index"]

model = SentenceTransformer('all-MiniLM-L6-v2')
app = FastAPI()

@app.get("/")
async def root():
    return {"message": "RAG Search API for arXiv cs.CL papers", "endpoint": "/search?q=your_query"}

@app.get("/search")
async def search(q: str, k: int = 3):
    """Search for top-k most relevant passages"""
    query_vector = model.encode([q])
    distances, indices = index.search(np.array(query_vector), k)
    
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "rank": i + 1,
            "distance": float(distances[0][i]),
            "text": all_chunks[idx]
        })
    
    return {"query": q, "num_results": len(results), "results": results}
'''

with open("main.py", "w") as f:
    f.write(app_code)

print("✅ FastAPI app saved to 'main.py'")
print("\nTo run the server:")
print("  uvicorn main:app --reload")
print("\nExample usage:")
print("  http://localhost:8000/search?q=transformer+models")

✅ FastAPI app saved to 'main.py'

To run the server:
  uvicorn main:app --reload

Example usage:
  http://localhost:8000/search?q=transformer+models


: 